# BambooAI Example Notebook

This notebook is a guided, end-to-end tour of BambooAI for analysis workflows.

**Note:** Cells that run the agent call an LLM and may incur cost. For parameter-by-parameter explanations and focused demos, see `bambooai.API.ipynb`.

## Setup

Expected working directory
- Run this notebook from the repo root where `bambooai_utils.py` and `testdata.csv` live.

Required vs optional
- `EXECUTION_MODE` is required by the wrapper.
- `LLM_CONFIG` is optional if `LLM_CONFIG.json` exists in the working directory.
- Provider keys depend on your LLM backend.

In [1]:
%load_ext autoreload
%autoreload 2

# System libraries.
import logging
import os
import random
import sys
from pathlib import Path

# Third party libraries.
import importlib.metadata as md
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Configure notebook plotting defaults.
# sns.set_style("whitegrid")
# plt.rcParams["figure.figsize"] = (12, 6)
# np.set_printoptions(suppress=True, precision=6)
# print("Notebook bootstrap complete.")

In [3]:
# Add local helper paths and import the notebook utilities.
# helpers_root_docker = Path("/app/helpers_root")
# helpers_root_local = Path.cwd() / "helpers_root"
# for candidate in [helpers_root_docker, helpers_root_local]:
#     if candidate.exists() and str(candidate) not in sys.path:
#         sys.path.insert(0, str(candidate))

# import bambooai
from bambooai import BambooAI

import bambooai_utils as butils
import helpers.hio as hio

ARTIFACTS_DIR = Path("artifacts")
print("Working directory:", Path.cwd())
print("bambooai version:", md.version("bambooai"))
# The project modules are now importable from the notebook.

Working directory: /git_root/tutorials/BambooAI
bambooai version: 0.4.24


In [4]:
# Initialize notebook logging through the shared utility module.
_LOG = logging.getLogger(__name__)
butils.init_logger(_LOG)
butils._setup_env()
print("Notebook logging initialized.")
# Logger output from the notebook and utility module now prints inline.

Notebook logging initialized.


## Sanity Check

Confirm the runtime configuration before starting any agent session.

In [16]:
os.environ['OPENAI_API_KEY']='sk-proj'
os.environ['GEMINI_API_KEY']=''

In [17]:
# Display the current execution and credential configuration.
execution_mode_env = os.getenv("EXECUTION_MODE", "<not set>")
llm_config_env = os.getenv("LLM_CONFIG", "<not set>")
llm_config_exists = Path("LLM_CONFIG.json").exists()
key_vars = ["OPENAI_API_KEY", "AZURE_OPENAI_API_KEY", "ANTHROPIC_API_KEY","GEMINI_API_KEY"]
present_keys = [key for key in key_vars if os.getenv(key)]

print("EXECUTION_MODE:", execution_mode_env)
print("LLM_CONFIG env:", llm_config_env)
print("LLM_CONFIG.json exists:", llm_config_exists)
print("Provider keys set for:", ", ".join(present_keys) or "<none>")
# This confirms whether the notebook has enough configuration to start BambooAI.

EXECUTION_MODE: local
LLM_CONFIG env: <not set>
LLM_CONFIG.json exists: True
Provider keys set for: OPENAI_API_KEY, GEMINI_API_KEY


## Data and Scenario

`testdata.csv` is a small synthetic customer dataset for demo analysis. It includes demographics, engagement metrics, and churn indicators.

Data dictionary
- user_id: Unique user identifier.
- age: User age.
- gender: User gender.
- country: Country code.
- device_type: Device type.
- signup_days_ago: Days since signup.
- sessions_last_30d: Sessions in the last 30 days.
- avg_session_duration_min: Average session duration in minutes.
- pages_per_session: Average pages per session.
- has_premium: Premium subscription indicator.
- monthly_spend_usd: Monthly spend in USD.
- support_tickets_90d: Support tickets in last 90 days.
- churned: Churn label.

In [8]:
# Create a small synthetic dataset if the demo CSV is missing.
def _create_testdata_if_missing(*, path: str = "testdata.csv") -> Path:
    """
    Create synthetic test data if the CSV is missing.

    :param path: output CSV path
    :return: path to the CSV file
    """
    csv_path = Path(path)
    if csv_path.exists():
        return csv_path
    random.seed(42)
    rows = []
    for idx in range(20):
        rows.append(
            {
                "user_id": 1001 + idx,
                "age": random.randint(18, 70),
                "gender": random.choice(["female", "male"]),
                "country": random.choice(["US", "CA", "DE", "IN"]),
                "device_type": random.choice(["mobile", "desktop", "tablet"]),
                "signup_days_ago": random.randint(1, 400),
                "sessions_last_30d": round(random.uniform(1, 30), 1),
                "avg_session_duration_min": round(random.uniform(1, 15), 2),
                "pages_per_session": round(random.uniform(1, 8), 2),
                "has_premium": random.choice([0, 1]),
                "monthly_spend_usd": round(random.uniform(5, 400), 2),
                "support_tickets_90d": random.randint(0, 5),
                "churned": random.choice([0, 1]),
            }
        )
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    return csv_path


csv_path = _create_testdata_if_missing(path="testdata.csv")
print("Dataset path:", csv_path)
# The demo dataset is available for the rest of the notebook.

Dataset path: testdata.csv


## Quick EDA

Take a quick look at the dataset before asking BambooAI questions about it.

In [9]:
# Load the dataframe and show the dataset dimensions.
df = butils._load_dataframe(butils._DEFAULT_CSV)
print("Shape:", df.shape)
display(df.dtypes.rename("dtype").to_frame())
# The dataframe loaded successfully and the schema is visible.

Shape: (500, 13)


,dtype
user_id,int64
age,int64
gender,str
country,str
device_type,str
signup_days_ago,int64
sessions_last_30d,float64
avg_session_duration_min,float64
pages_per_session,float64
has_premium,int64


In [10]:
# Summarize missing values and preview the first rows.
display(df.isna().sum().rename("missing_values").to_frame())
display(df.head())
# The dataset appears ready for interactive analysis.

,missing_values
user_id,0
age,0
gender,0
country,0
device_type,0
signup_days_ago,0
sessions_last_30d,0
avg_session_duration_min,20
pages_per_session,20
has_premium,0


,user_id,age,gender,country,device_type,signup_days_ago,sessions_last_30d,avg_session_duration_min,pages_per_session,has_premium,monthly_spend_usd,support_tickets_90d,churned
0,1001,56,female,IN,tablet,169,16.0,4.029,3.98,1,387.378,2,0
1,1002,69,female,CA,mobile,217,6.4,8.126,5.76,0,8.040,0,1
2,1003,46,female,US,mobile,378,13.0,13.530,5.60,0,52.960,2,0
3,1004,32,female,US,desktop,119,12.0,20.280,5.26,1,90.864,0,0
4,1005,60,male,DE,desktop,190,9.0,5.338,2.96,1,316.692,0,0


## Conversation Loop

`butils._run_agent(...)` an interactive chat loop.
Type `exit` or `quit` when you are done, or interrupt the kernel to stop.

Try these prompts and what to expect
- Summarize columns, types, and missing values. Expect a schema summary.
- Show top 5 rows and a brief dataset description. Expect a quick preview.
- Plot distribution of monthly_spend_usd. Expect a histogram.
- Compare churn rate by has_premium. Expect a grouped summary.
- Identify outliers in avg_session_duration_min. Expect a potential outlier list.

In [12]:
# Resolve the execution mode for the notebook session.
args = butils._parse().parse_args([])
execution_mode = butils._resolve_execution_mode(
    args.execution_mode or os.getenv("EXECUTION_MODE", "local")
)
os.environ["EXECUTION_MODE"] = execution_mode
print("Execution mode:", execution_mode)
# The notebook session now has an explicit execution mode.

Execution mode: local


In [19]:
# Build the minimal BambooAI configuration.
minimal_config = {
    "planning": False, #No planning enabled
    "vector_db": False, #No vector DB searches 
    "search_tool": False, #No web searche enabled
}
display(pd.Series(minimal_config, name="enabled").to_frame())
# This is the smallest configuration that still exercises the core workflow.

,enabled
planning,False
vector_db,False
search_tool,False


In [14]:
# Construct the minimal BambooAI agent and show its type.
bamboo_agent = butils._build_bamboo_agent(df, **minimal_config)
print("Constructed agent type:", type(bamboo_agent).__name__)
# The minimal BambooAI agent is ready for interaction.

Constructed agent type: BambooAI


In [ ]:
# Start the minimal config conversation loop.
butils._run_agent(bamboo_agent)
# The minimal config agent interactive session is now running.

In [ ]:
# Construct the planning-enabled BambooAI agent.
bamboo_planning = butils._build_bamboo_agent(
    df,
    planning=True,
    vector_db=False,
    search_tool=False,
)
print("Constructed planning agent type:", type(bamboo_planning).__name__)
# The planning-enabled agent is ready for interaction.

In [ ]:
# Start the planning-enabled conversation loop.
butils._run_agent(bamboo_planning)
# The planning-enabled interactive session is now running.

## Semantic Search Demo

Create an auxiliary dataset and run BambooAI with semantic search features enabled.

In [ ]:
# Create the auxiliary dataset used by the semantic-search configuration.
hio.create_dir(str(ARTIFACTS_DIR), incremental=True)
aux_path = ARTIFACTS_DIR / "auxiliary_demo.csv"
aux_df = pd.DataFrame(
    {
        "country": ["US", "CA", "DE"],
        "region_label": ["North America", "North America", "Europe"],
    }
)
aux_df.to_csv(aux_path, index=False)
display(aux_df)
print("Wrote auxiliary dataset:", aux_path)
# The semantic-search demo now has an auxiliary dataset to join against.

In [ ]:
# Build the semantic-search BambooAI agent.
semantic_config = {
    "planning": True,
    "vector_db": True,
    "search_tool": True,
    "auxiliary_datasets": [str(aux_path)],
}
display(pd.Series(semantic_config, name="value").to_frame())
bamboo_semantic = BambooAI(df=df, **semantic_config)
print("Constructed semantic agent type:", type(bamboo_semantic).__name__)
# The semantic-search configuration is ready for interaction.

In [ ]:
# Start the semantic-search conversation loop.
butils._run_agent(bamboo_semantic)
# The semantic-search interactive session is now running.

## Ontology Demo

Create a small ontology file and run BambooAI with ontology grounding enabled.

In [ ]:
# Write a minimal ontology file for the dataframe fields.
hio.create_dir(str(ARTIFACTS_DIR), incremental=True)
ontology_path = ARTIFACTS_DIR / "mini_ontology.ttl"
ontology_path.write_text(
    "@prefix ex: <http://example.com/> .\n"
    "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n"
    "@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n"
    "ex:Customer a rdfs:Class .\n"
    "ex:churned a rdfs:Property ;\n"
    "  rdfs:domain ex:Customer ;\n"
    "  rdfs:range xsd:boolean ;\n"
    '  rdfs:label "churned" .\n'
    "ex:monthly_spend_usd a rdfs:Property ;\n"
    "  rdfs:domain ex:Customer ;\n"
    "  rdfs:range xsd:decimal ;\n"
    '  rdfs:label "monthly_spend_usd" .\n'
    "ex:has_premium a rdfs:Property ;\n"
    "  rdfs:domain ex:Customer ;\n"
    "  rdfs:range xsd:boolean ;\n"
    '  rdfs:label "has_premium" .\n'
)
print(ontology_path.read_text())
# The ontology file is now available for grounding dataframe questions.

In [ ]:
# Build the ontology-grounded BambooAI agent.
ontology_config = {
    "planning": True,
    "exploratory": True,
    "df_ontology": str(ontology_path),
}
display(pd.Series(ontology_config, name="value").to_frame())
bamboo_ontology = BambooAI(df=df, **ontology_config)
print("Constructed ontology agent type:", type(bamboo_ontology).__name__)
# The ontology-grounded configuration is ready for interaction.

In [ ]:
# Start the ontology-grounded conversation loop.
butils._run_agent(bamboo_ontology)
# The ontology-grounded interactive session is now running.

## Custom Prompt Demo

Create a custom prompt file and run BambooAI with custom prompts enabled.

In [ ]:
# Write a small custom prompt file for the demo run.
hio.create_dir(str(ARTIFACTS_DIR), incremental=True)
custom_prompt_path = ARTIFACTS_DIR / "custom_prompts.yaml"
custom_prompt_path.write_text(
    "# Placeholder prompts for BambooAI\n"
    'planner_prompt: "You are a careful planner."\n'
    'code_prompt: "Write concise pandas code."\n'
)
print(custom_prompt_path.read_text())
# The custom prompt file is available for the next BambooAI run.

In [ ]:
# Build the custom-prompt BambooAI agent.
custom_prompt_config = {
    "planning": False,
    "exploratory": True,
    "custom_prompt_file": str(custom_prompt_path),
}
display(pd.Series(custom_prompt_config, name="value").to_frame())
bamboo_custom = BambooAI(df=df, **custom_prompt_config)
print("Constructed custom prompt agent type:", type(bamboo_custom).__name__)
# The custom-prompt configuration is ready for interaction.

In [ ]:
# Start the custom-prompt conversation loop.
butils._run_agent(bamboo_custom)
# The custom-prompt interactive session is now running.

## Full Featured Run

This run combines planning, semantic search, ontology grounding, and custom prompts.
It expects the artifacts created in the feature sections above.

Curated prompts and expected behavior
- Summarize columns, types, missing percent, and show `df.head()`.
- What factors correlate most with churn.
- Add region labels to country and summarize churn by region.
- Explain valid values for `churned` and `has_premium`.
- Provide a concise bullet summary with 3 takeaways.

In [ ]:
# Locate the optional artifacts that enrich the full BambooAI run.
aux_path = ARTIFACTS_DIR / "auxiliary_demo.csv"
ontology_path = ARTIFACTS_DIR / "mini_ontology.ttl"
custom_prompt_path = ARTIFACTS_DIR / "custom_prompts.yaml"
artifact_status = pd.Series(
    {
        "auxiliary_demo.csv": aux_path.exists(),
        "mini_ontology.ttl": ontology_path.exists(),
        "custom_prompts.yaml": custom_prompt_path.exists(),
    },
    name="exists",
)
display(artifact_status.to_frame())
# This shows which optional artifacts are available for the combined run.

In [ ]:
# Assemble the full-feature BambooAI configuration from the available artifacts.
full_config = {
    "planning": True,
    "vector_db": True,
    "search_tool": True,
    "exploratory": True,
}
if aux_path.exists():
    full_config["auxiliary_datasets"] = [str(aux_path)]
if ontology_path.exists():
    full_config["df_ontology"] = str(ontology_path)
if custom_prompt_path.exists():
    full_config["custom_prompt_file"] = str(custom_prompt_path)

display(pd.Series(full_config, name="value").to_frame())
# The combined configuration is ready to instantiate.

In [ ]:
# Build the full-feature BambooAI agent.
bamboo_full = BambooAI(df=df, **full_config)
print("Constructed full agent type:", type(bamboo_full).__name__)
# The full-feature BambooAI agent is ready for interaction.

In [ ]:
# Start the full-feature conversation loop.
butils._run_agent(bamboo_full)
# The full-feature interactive session is now running.

## Troubleshooting

Missing env vars
- Ensure `EXECUTION_MODE` is set in `.env` or environment.
- Ensure provider keys are set for your LLM backend.

Missing files or wrong working directory
- Run the notebook from the repo root.
- Re-run the data creation cell to regenerate missing files.

Import errors
- Verify BambooAI and pandas are installed in this environment.
- Restart the kernel after changing your environment.

Agent hangs or no output
- Confirm network access to your LLM backend.
- Check logs for rate limits or authentication errors.
- Try the minimal quickstart run to isolate failures.

## Cleanup

Remove the generated artifacts if you want to reset the demo state.

In [20]:
# Delete the generated artifacts from the notebook run.
for path in [
    ARTIFACTS_DIR / "auxiliary_demo.csv",
    ARTIFACTS_DIR / "mini_ontology.ttl",
    ARTIFACTS_DIR / "custom_prompts.yaml",
]:
    if path.exists():
        path.unlink()
        print("Deleted:", path)
    else:
        print("Not found:", path)
# The generated files have been removed if they existed.

Deleted: artifacts/auxiliary_demo.csv
Deleted: artifacts/mini_ontology.ttl
Deleted: artifacts/custom_prompts.yaml


In [21]:
# Remove the artifact directory if it is now empty.
if ARTIFACTS_DIR.exists() and not any(ARTIFACTS_DIR.iterdir()):
    ARTIFACTS_DIR.rmdir()
    print("Removed empty directory:", ARTIFACTS_DIR)
else:
    print("Artifact directory still contains files:", ARTIFACTS_DIR)
# The artifact directory state is now explicit.

Removed empty directory: artifacts
